In [1]:
import torch
import torch.optim as optim

In [2]:
from pathlib import Path

In [4]:
from config import BaseConfig
from config import config as cfg
from datasets import AudioSpectrogramDataset
from utils.common import split_data, train_vae
from models.vae import VAE
from models.pca_baseline import PCABaseline

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
npy_dir = Path("../..") / cfg.FEATURES_DIR

In [7]:
dataset = AudioSpectrogramDataset(dataset_dir=npy_dir)
train_loader, test_loader = split_data(dataset=dataset, ratio=0.8, batch_size=cfg.BATCH_SIZE[2], shuffle=cfg.SHUFFLE)

### Training Script

In [ ]:
# # basic_vae = VAE(cfg=config, model_type="basic").to(device)
# # print(basic_vae)
# # optimizer = optim.Adam(basic_vae.parameters(), lr=config.LR)
# # history = train_vae(model=basic_vae, train_loader=train_loader, test_loader=test_loader, optimizer=optimizer, epochs=10, beta=config.BETA, beta_type="annealing", device=device)

# # Replaces your original code entirely
# pca_baseline = PCABaseline(variance_threshold=0.90, n_components=32)
# pca_baseline.fit(train_loader, test_loader)

# # Compare reconstruction error against your VAE recon loss
# train_error = pca_baseline.reconstruction_error(train_loader)
# test_error  = pca_baseline.reconstruction_error(test_loader)
# print(f"PCA Train Recon MSE: {train_error:.4f}")
# print(f"PCA Test  Recon MSE: {test_error:.4f}")

### Optuna Code

In [15]:
import optuna
from optuna.samplers import TPESampler

### Separate search spaces for each model type (basic, convolutional, beta, conditional)

In [9]:
# try SimpleNamespace --> does not override original config object
# TODO: can I make sure h1 > h2 > latent?

def _suggest_basic_vae(trial: optuna.trial.Trial):
    base_cfg = BaseConfig()
    base_cfg.LATENT_DIM = trial.suggest_categorical("latent_dim", [16, 32, 64, 128, 256])
    base_cfg.HIDDEN_DIM_1 = trial.suggest_categorical("hidden_dim_1", [256, 512, 1024])
    base_cfg.HIDDEN_DIM_2 = trial.suggest_categorical("hidden_dim_2", [64, 128, 256])
    
    return base_cfg

In [10]:
SEARCH_SPACES = {
    "basic": _suggest_basic_vae
}

In [24]:
def _suggest_shared_parameters(trial: optuna.trial.Trial):
    base_cfg = BaseConfig()
    base_cfg.LR = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    base_cfg.BATCH_SIZE = trial.suggest_categorical("batch_size", [16, 32, 64])
    
    return base_cfg

In [33]:
def make_objective(model_type, dataset, device, epochs=30):
    if model_type not in SEARCH_SPACES: raise ValueError(f"Unknown model_type: `{model_type}`. Better luck next time.")
    
    def objective(trial):
        # model-specific hyperparameters
        trial_cfg = SEARCH_SPACES[model_type](trial)
        
        # shared hyperparameters
        shared_cfg = _suggest_shared_parameters(trial)
        lr = shared_cfg.LR
        batch_size = shared_cfg.BATCH_SIZE
        
        train_loader, test_loader = split_data(dataset=dataset, ratio=0.8, batch_size=batch_size, shuffle=trial_cfg.SHUFFLE)
        
        model = VAE(cfg=trial_cfg, model_type=model_type).to(device=device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        history = train_vae(
            model=model,
            train_loader=train_loader,
            test_loader=test_loader,
            optimizer=optimizer,
            epochs=epochs, 
            beta_type="fixed",
            device=device
        )
        
        # taking the scores from the last test iteration for evaluation??
        return 0.6 * history["test_recon"][-1] + 0.4 * history["test_kl"][-1]
    
    return objective

In [28]:
def run_tuning(model_type, dataset, device, n_trials=30, epochs=30):
    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=42),
        study_name=f"VAE_{model_type}_Tuning"
    )
    study.optimize(
        make_objective(model_type=model_type, dataset=dataset, device=device, epochs=epochs),
        n_trials=n_trials,
        show_progress_bar=True
    )
    
    print(f"Best trial for `{model_type}`:\nScore: {study.best_trial.value:.4f}")
    for k, v in study.best_trial.params.items():
        print(f"    {k:<25} {v}")
        
    return study

In [29]:
def compare_studies(studies: dict):
    """
    Accept any number of studies as a dict of {label: study}.
    Example: compare_studies({"basic": study1, "advanced": study2})
    """
    labels = list(studies.keys())
    all_keys = sorted(set(k for s in studies.values() for k in s.best_trial.params))

    col_w = 20
    header = f"{'Param':<25}" + "".join(f"{l:<{col_w}}" for l in labels)
    print(f"\n{header}")
    print("-" * (25 + col_w * len(labels)))

    for k in all_keys:
        row = f"{k:<25}"
        for s in studies.values():
            v = s.best_trial.params.get(k, "N/A")
            row += f"{str(v):<{col_w}}"
        print(row)

    print("-" * (25 + col_w * len(labels)))
    score_row = f"{'Best score':<25}"
    for s in studies.values():
        score_row += f"{s.best_trial.value:<{col_w}.4f}"
    print(score_row)

In [34]:
study_basic = run_tuning(model_type="basic", dataset=dataset, device=device)
study_advanced = run_tuning(model_type="advanced", dataset=dataset, device=device)

compare_studies({
    "basic":    study_basic,
    "advanced": study_advanced,
})

[I 2026-03-28 20:03:23,497] A new study created in memory with name: VAE_basic_Tuning
  0%|          | 0/30 [00:00<?, ?it/s]

--------------------------------------------------
Epoch 1 / 30
Metric       | Train        | Test        
--------------------------------------------------
Total Loss   | 48550124093340517587004424192.0000 | 823132702010882844126085120.0000
Recon        | 45752264280379827071848808448.0000 | 541846733598690844906356736.0000
KL Div       | 2797858972846612873429909504.0000 | 281285970865482816253067264.0000
--------------------------------------------------

--------------------------------------------------
Epoch 2 / 30
Metric       | Train        | Test        
--------------------------------------------------
Total Loss   | 72959023344429929390416068608.0000 | 9348689154447589425853497344.0000
Recon        | 70157547454548692823571431424.0000 | 9067403010190368518560546816.0000
KL Div       | 2801473947454399383278190592.0000 | 281285970865482816253067264.0000
--------------------------------------------------

--------------------------------------------------
Epoch 3 / 30
Metric

  3%|▎         | 1/30 [00:27<13:28, 27.86s/it]

--------------------------------------------------
Epoch 30 / 30
Metric       | Train        | Test        
--------------------------------------------------
Total Loss   | nan          | nan         
Recon        | nan          | nan         
KL Div       | nan          | nan         
--------------------------------------------------

[W 2026-03-28 20:03:51,361] Trial 0 failed with parameters: {'latent_dim': 32, 'hidden_dim_1': 1024, 'hidden_dim_2': 128, 'lr': 0.008706020878304856, 'batch_size': 16} because of the following error: The value nan is not acceptable.
[W 2026-03-28 20:03:51,363] Trial 0 failed with value nan.
--------------------------------------------------
Epoch 1 / 30
Metric       | Train        | Test        
--------------------------------------------------
Total Loss   | 1204.5367    | 344.4357    
Recon        | 1166.1671    | 314.9316    
KL Div       | 38.3696      | 29.5041     
--------------------------------------------------

-----------------------------

Best trial: 1. Best value: 55.4251:   7%|▋         | 2/30 [00:40<08:56, 19.17s/it]

...

Best trial: 28. Best value: 39.9232:  97%|█████████▋| 29/30 [10:29<00:31, 31.01s/it]

--------------------------------------------------
Epoch 30 / 30
Metric       | Train        | Test        
--------------------------------------------------
Total Loss   | 72.9419      | 69.7649     
Recon        | 63.1495      | 60.0860     
KL Div       | 9.7924       | 9.6790      
--------------------------------------------------

[I 2026-03-28 20:13:52,849] Trial 28 finished with value: 39.923163970947265 and parameters: {'latent_dim': 16, 'hidden_dim_1': 512, 'hidden_dim_2': 256, 'lr': 0.00047128915005434553, 'batch_size': 16}. Best is trial 28 with value: 39.923163970947265.
--------------------------------------------------
Epoch 1 / 30
Metric       | Train        | Test        
--------------------------------------------------
Total Loss   | 359.8925     | 121.2687    
Recon        | 333.2231     | 98.5250     
KL Div       | 26.6694      | 22.7437     
--------------------------------------------------

--------------------------------------------------
Epoch 2 / 30
Metri

Best trial: 28. Best value: 39.9232: 100%|██████████| 30/30 [10:58<00:00, 21.96s/it]
[I 2026-03-28 20:14:22,356] A new study created in memory with name: VAE_advanced_Tuning


--------------------------------------------------
Epoch 30 / 30
Metric       | Train        | Test        
--------------------------------------------------
Total Loss   | 70.6036      | 73.4780     
Recon        | 60.7702      | 63.5315     
KL Div       | 9.8334       | 9.9466      
--------------------------------------------------

[I 2026-03-28 20:14:22,352] Trial 29 finished with value: 42.09751486206054 and parameters: {'latent_dim': 16, 'hidden_dim_1': 512, 'hidden_dim_2': 256, 'lr': 0.0003866498612570256, 'batch_size': 16}. Best is trial 28 with value: 39.923163970947265.
Best trial for `basic`:
Score: 39.9232
    latent_dim                16
    hidden_dim_1              512
    hidden_dim_2              256
    lr                        0.00047128915005434553
    batch_size                16


ValueError: Unknown model_type: `advanced`. Better luck next time.

In [36]:
study_basic.best_trial.params

{'latent_dim': 16,
 'hidden_dim_1': 512,
 'hidden_dim_2': 256,
 'lr': 0.00047128915005434553,
 'batch_size': 16}

In [41]:
study_basic.best_trial.value

39.923163970947265

In [69]:
from optuna.visualization import plot_param_importances

In [64]:
! pip install plotly


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [71]:
study_basic

In [72]:
study_basic.best_params

{'latent_dim': 16,
 'hidden_dim_1': 512,
 'hidden_dim_2': 256,
 'lr': 0.00047128915005434553,
 'batch_size': 16}

In [70]:
import plotly
print(plotly.__version__)
plot_param_importances(study=study_basic).show()

6.6.0


ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

In [ ]:
study_basic.